In [2]:
!pip install cloud-tpu-client==0.10 torch==2.0.0 torchvision==0.15.1 

ERROR: Could not find a version that satisfies the requirement torch==2.0.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.0.0


In [3]:
!pip install torch transformers datasets trl accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 16.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 25.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=

In [4]:
import datasets
print(datasets.__file__)

/usr/local/lib/python3.12/dist-packages/datasets/__init__.py


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
import os
import torch
from trl import DPOTrainer
base_path=r"/kaggle/input/datasets/mostafanasr14/dpo-dataset/dpo_training"
print("="*50)
print("DPO ALIGNMENT - Fine-Tuning (LoRA)")
print("="*50)

# --- Configuration ---
BASE_MODEL_PATH = os.path.join(base_path, "output", "sft_model_merged")
MAX_SEQ_LENGTH = 256  
DATASET_PATH = os.path.join(base_path, "datasets", "dpo_dataset")
OUTPUT_DIR = os.path.join(base_path, "output", "dpo_model_full")



DPO ALIGNMENT - Fine-Tuning (LoRA)


In [6]:
# --- Model & Tokenizer ---
print(f"Loading merged SFT model from: {BASE_MODEL_PATH}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token
if tokenizer.unk_token is None:
    tokenizer.unk_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# Enable gradient checkpointing to drastically reduce memory usage during full fine-tuning
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading merged SFT model from: /kaggle/input/datasets/mostafanasr14/dpo-dataset/dpo_training/output/sft_model_merged


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
import os
from datasets import load_from_disk, disable_caching

# 1. Kill the disk caching globally
disable_caching()

# 2. Set a writable home for any persistent metadata
os.environ["HF_HOME"] = "/kaggle/working/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/dataset_cache"

print("Loading dataset...")
dataset = load_from_disk(DATASET_PATH)

def format_dpo_fn(example):
    prompt_text = f"<|im_start|>system\nYou are a helpful assistant specialized in math reasoning.<|im_end|>\n<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n"
    chosen_text = f"{example['chosen_response']}<|im_end|>"
    rejected_text = f"{example['rejected_response']}<|im_end|>"
    
    return {
        "prompt": prompt_text,
        "chosen": chosen_text,
        "rejected": rejected_text,
    }

print("Formatting dataset for DPO (Forcing Memory-Only)...")
# keep_in_memory=True is the key here to avoid the /kaggle/input folder
formatted_ds = dataset.map(
    format_dpo_fn, 
    remove_columns=dataset.column_names,
    keep_in_memory=True,
    load_from_cache_file=False
)

print(f"Dataset ready. Total samples: {len(formatted_ds)}")

Loading dataset...
Formatting dataset for DPO (Forcing Memory-Only)...


Map:   0%|          | 0/2418 [00:00<?, ? examples/s]

Dataset ready. Total samples: 2418


In [8]:
# from datasets import load_from_disk  # <-- Added this import

# # --- Data Preparation ---
# print("Loading dataset...")
# # Ensure DATASET_PATH points to the directory containing 'dataset_info.json'
# dataset = load_from_disk(DATASET_PATH)

# def format_dpo_fn(example):
#     # ChatML format for Math Reasoning
#     prompt_text = (
#         f"<|im_start|>system\n"
#         f"You are a helpful assistant specialized in math reasoning.<|im_end|>\n"
#         f"<|im_start|>user\n"
#         f"{example['instruction']}<|im_end|>\n"
#         f"<|im_start|>assistant\n"
#     )
    
#     # We strip potential leading spaces to ensure the model doesn't 
#     # learn to start every answer with a space after the assistant tag
#     chosen_text = f"{example['chosen_response'].strip()}<|im_end|>"
#     rejected_text = f"{example['rejected_response'].strip()}<|im_end|>"
    
#     return {
#         "prompt": prompt_text,
#         "chosen": chosen_text,
#         "rejected": rejected_text,
#     }

# print("Formatting dataset for DPO...")
# # Use num_proc to speed this up if your dataset is large
# formatted_ds = dataset.map(
#     format_dpo_fn, 
#     remove_columns=dataset.column_names,
#     desc="Applying ChatML Template"
# )

# print(f"Dataset ready. Total samples: {len(formatted_ds)}")
# # Quick sanity check: print the first example
# print("\n--- Sample Prompt ---")
# print(formatted_ds[0]['prompt'])

In [9]:
from trl import DPOConfig, DPOTrainer

# training_args = DPOConfig(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=5e-6,
#     num_train_epochs=1,
#     # --- Critical for Visibility ---
#     logging_strategy="steps",    # Tell it to log based on steps
#     logging_steps=3,             # Log every single step (best for slow CPU training)
#     log_level="info",            # Ensure the library talks to you
#     report_to="none",            # Keeps it in the console, not WandB/Tensorboard
#     # ------------------------------
#     gradient_checkpointing=False,  
#     use_cpu=True,
#     bf16=False,
#     fp16=False,
#     max_length=256,
#     beta=0.1,
#     remove_unused_columns=False,
# )
OUTPUT_model = r"/kaggle/working"


training_args = DPOConfig(
    output_dir=OUTPUT_model,
    per_device_train_batch_size=2,     
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    num_train_epochs=6,
    logging_strategy="steps",
    logging_steps=100,                   
    log_level="info",
    report_to="none",
    
    # --- TPU Specific Adjustments ---
    save_strategy="steps",
    save_steps=150,               # Save a checkpoint every 150 steps
    save_total_limit=2,          
    use_cpu=False,               
    bf16=True,                   
    fp16=False,
    
    # --- Memory & Optimization ---
    gradient_checkpointing=True,       
    max_length=1024,
    beta=0.1,
    remove_unused_columns=False,
    
    # --- Distributed/TPU Settings ---
    # When using Kaggle TPU, Accelerate usually handles the 'tpu' flag via CLI, 
    # but in a notebook, the trainer will detect the XLA device automatically.
)

In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Define the LoRA Configuration
peft_config = LoraConfig(
    r=32,                    # Rank: higher = more parameters, 16 is a good balance
    lora_alpha=64,           # Scaling factor (usually 2x Rank)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Targets the attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 2. Wrap your existing model
model = get_peft_model(model, peft_config)

# 3. Print the new parameter count to verify the 10% goal
model.print_trainable_parameters()

trainable params: 4,325,376 || all params: 498,358,144 || trainable%: 0.8679


In [11]:
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_ds,
    processing_class=tokenizer,
)


Adding EOS to train dataset:   0%|          | 0/2418 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2418 [00:00<?, ? examples/s]

In [12]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None
if os.path.exists(OUTPUT_model):
    last_checkpoint = get_last_checkpoint(OUTPUT_model)
    if last_checkpoint is not None:
        print(f"Found checkpoint: {last_checkpoint}")



In [17]:
import warnings 
warnings.filterwarnings("ignore")


In [18]:
print("Starting Full DPO training...")
# Start training!
if last_checkpoint is not None:
    print(f"Resuming training from {last_checkpoint}...")
dpo_trainer.train(resume_from_checkpoint=last_checkpoint)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151645}.


Starting Full DPO training...


***** Running training *****
  Num examples = 2,418
  Num Epochs = 6
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 2
  Total optimization steps = 3,630
  Number of trainable parameters = 4,325,376


Step,Training Loss
100,0.691007
200,0.684825
300,0.691979
400,0.691108
500,0.691457
600,0.689193
700,0.600874
800,0.598520
900,0.592683
1000,0.582664


Saving model checkpoint to /kaggle/working/checkpoint-150
loading configuration file /kaggle/input/datasets/mostafanasr14/dpo-dataset/dpo_training/output/sft_model_merged/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

TrainOutput(global_step=3630, training_loss=0.4973329347027235, metrics={'train_runtime': 11369.6274, 'train_samples_per_second': 1.276, 'train_steps_per_second': 0.319, 'total_flos': 2.635518588460032e+16, 'train_loss': 0.4973329347027235})

In [19]:
OUTPUT_model = r"/kaggle/working"
# --- Save ---
print("Saving DPO model...")
model.save_pretrained(OUTPUT_model)
tokenizer.save_pretrained(OUTPUT_model)
print(f"✓ Full DPO Model saved to {OUTPUT_model}")


loading configuration file /kaggle/input/datasets/mostafanasr14/dpo-dataset/dpo_training/output/sft_model_merged/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max

Saving DPO model...
✓ Full DPO Model saved to /kaggle/working
